# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
import os
import tempfile
import whisper
import gradio as gr

from gtts import gTTS
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)

True

In [3]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("Groq Loaded")
print("Gemini Loaded")


Groq Loaded
Gemini Loaded


In [4]:
# =========================================================
# GROQ CLIENT
# =========================================================

groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

# =========================================================
# GEMINI CLIENT
# =========================================================

gemini_client = OpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [5]:
MODELS = {
    
    # =====================================================
    # GROQ MODELS
    # =====================================================
    
    "Groq Llama 70B": {
        "provider": "groq",
        "model": "llama-3.3-70b-versatile"
    },
    
    "Groq Gemma 9B": {
        "provider": "groq",
        "model": "gemma2-9b-it"
    },
    
    # =====================================================
    # GEMINI MODEL
    # =====================================================
    
    "Gemini Flash Lite": {
        "provider": "gemini",
        "model": "gemini-2.5-flash-lite"
    },
    
}

In [6]:
# tiny / base / small / medium

whisper_model = whisper.load_model("base")

In [7]:
SYSTEM_PROMPT = """
You are an expert technical tutor and AI engineer.

Your responsibilities:
- Explain concepts step-by-step
- Use beginner-friendly language
- Teach clearly and logically
- Give examples
- Explain intuition before code
- Provide optimized solutions
- Use markdown formatting

For coding questions:
1. Explain intuition
2. Explain brute force approach
3. Explain optimized approach
4. Then provide code
5. Mention time complexity

Always keep responses structured and clean.
"""

In [8]:
def analyze_question(question):

    words = len(question.split())
    chars = len(question)

    result = f"""
## Question Analysis Tool

- Total Words: {words}
- Total Characters: {chars}

"""

    return result

In [9]:
def speech_to_text(audio_path):

    # ============================================
    # CHECK AUDIO
    # ============================================

    if audio_path is None:
        return "No audio detected. Please record again."

    try:

        result = whisper_model.transcribe(audio_path)

        text = result["text"]

        return text

    except Exception as e:

        return f"Speech recognition error: {str(e)}"

In [10]:
def generate_answer(question, model_name, use_tool):

    model_info = MODELS[model_name]

    provider = model_info["provider"]

    model = model_info["model"]

    # =================================================
    # SELECT CLIENT
    # =================================================

    if provider == "groq":
        client = groq_client

    elif provider == "gemini":
        client = gemini_client

    # =================================================
    # TOOL
    # =================================================

    tool_output = ""

    if use_tool:
        tool_output = analyze_question(question)

    # =================================================
    # MESSAGES
    # =================================================

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question + "\\n\\n" + tool_output
        }
    ]

    # =================================================
    # API CALL
    # =================================================

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
        stream=True
    )

    partial_response = ""

    for chunk in stream:

        delta = chunk.choices[0].delta.content

        if delta:
            partial_response += delta

    return partial_response

In [16]:
import re

def clean_text_for_speech(text):

    # Remove markdown headings
    text = re.sub(r'#', '', text)

    # Remove bullet points
    text = re.sub(r'\\*', '', text)
    text = re.sub(r'-', ' ', text)

    # Remove backticks
    text = re.sub(r'`', '', text)

    # Remove multiple newlines
    text = re.sub(r'\\n+', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\\s+', ' ', text)

    return text.strip()

In [17]:
def text_to_speech(text):

    # =========================================
    # CLEAN TEXT
    # =========================================

    clean_text = clean_text_for_speech(text)

    # =========================================
    # TEXT TO SPEECH
    # =========================================

    tts = gTTS(text=clean_text)

    temp_audio = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp3"
    )

    tts.save(temp_audio.name)

    return temp_audio.name

In [18]:
def text_chat(question, model_name, use_tool):

    answer = generate_answer(
        question,
        model_name,
        use_tool
    )

    audio_file = text_to_speech(answer)

    return answer, audio_file

In [19]:
def full_voice_chat(audio_path, model_name, use_tool):

    # ============================================
    # CHECK AUDIO INPUT
    # ============================================

    if audio_path is None:

        return (
            "No speech detected",
            "Please record your voice properly.",
            None
        )

    # ============================================
    # SPEECH TO TEXT
    # ============================================

    question = speech_to_text(audio_path)

    # ============================================
    # HANDLE ERROR
    # ============================================

    if "error" in question.lower():

        return (
            question,
            "Speech recognition failed.",
            None
        )

    # ============================================
    # GENERATE ANSWER
    # ============================================

    answer = generate_answer(
        question,
        model_name,
        use_tool
    )

    # ============================================
    # TEXT TO SPEECH
    # ============================================

    audio_file = text_to_speech(answer)

    return (
        question,
        answer,
        audio_file
    )

In [20]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # Voice AI Technical Tutor

        ## Features

        ✅ Voice Input  
        ✅ Voice Output  
        ✅ Whisper Speech-To-Text  
        ✅ Multi-Model Support  
        ✅ Tool Usage  
        ✅ AI Technical Tutor  
        """
    )

    # =================================================
    # SETTINGS
    # =================================================

    with gr.Row():

        model_dropdown = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="Groq Llama 70B",
            label="Choose Model"
        )

        tool_checkbox = gr.Checkbox(
            label="Use Tool",
            value=True
        )

    # =================================================
    # TEXT CHAT SECTION
    # =================================================

    gr.Markdown("## Text Chat")

    text_question = gr.Textbox(
        label="Ask Question",
        placeholder="Explain transformers...",
        lines=5
    )

    text_button = gr.Button("Ask With Text")

    text_output = gr.Markdown()

    text_audio_output = gr.Audio(
        label="AI Voice Response",
        autoplay=True
    )

    text_button.click(
        fn=text_chat,
        inputs=[
            text_question,
            model_dropdown,
            tool_checkbox
        ],
        outputs=[
            text_output,
            text_audio_output
        ]
    )

    # =================================================
    # VOICE CHAT SECTION
    # =================================================

    gr.Markdown("## Voice Chat")

    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="Speak Here",
        waveform_options=gr.WaveformOptions(
            waveform_color="#01C6FF",
            waveform_progress_color="#0066B4",
            skip_length=2,
            show_controls=True
        )
    )

    voice_button = gr.Button("Ask With Voice")

    recognized_text = gr.Textbox(
        label="Recognized Speech"
    )

    voice_output = gr.Markdown()

    voice_audio_output = gr.Audio(
        label="AI Voice Response",
        autoplay=True
    )

    voice_button.click(
        fn=full_voice_chat,
        inputs=[
            audio_input,
            model_dropdown,
            tool_checkbox
        ],
        outputs=[
            recognized_text,
            voice_output,
            voice_audio_output
        ]
    )

c:\Users\KIIT\projects\llm_engineering\.venv\Lib\site-packages\gradio\components\audio.py:200: UserWarning: The `show_controls` parameter is deprecated and will be removed in a future release. Use `show_recording_waveform` instead.
  warnings.warn(


In [21]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Users\KIIT\projects\llm_engineering\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
